# Trimet Data Project

For our project, we decided to work with Trimet data as it sounded interesting and there seem and their developer portal have very good documentation for accessing data.

There were 2 static data GIS (Geospatial Data) and GFTS Static (General Transit Feed Specification).  The GIS basically provides an outline of the Transit map such as stop locations, route boundaries, garage locations and GFTS Static mainly contains text files like `stops.txt` `stop_times.txt` which shows how trimet vehicles are planned to move.

The main interesting we found that had lots of data was GFTS Realtime.  This logs information on Trimet vehicles in realtime giving back information like location using longitude and latitude, whether the vehicle is delayed or early, what its next stop and previous stop is and sometimes even an approximate of how full the vehicle is.

We worked with this api and built a collector in python to collect data on vehicles every minute.  We ran the collector program as much as we can as you cannot have too much data. The data we recieved from the api is formated in json which we converted into a csv.  In this conversion to csv, we only selected specific columns which sounded interesting in order to minimize the size of the csv file.



## Data Processing

Trimet provided documentation for an api called [/vehicles](https://developer.trimet.org/ws_docs/vehicle_locations_ws.shtml) showing information on what data is collected for a vehicle in real time

Example Json data for 1 vehicle.  This is collected every minute.

*Every minute, it logs about 400 vehicles during the day and almost none at night.*

```
  {
    "routeColor": "61A744",            // Color for route, can be dropped since redundant
    "expires": 1760993085102,          // Used by trimet to ensure data up to date
    "signMessage": "FX2 To Gresham",   // display message
    "serviceDate": 1760943600000,      // Can be dropped since we know the date from collection date
    "loadPercentage": 15,              // Estimated passenger load (0–100%)
    "latitude": 45.503744191408664,    
    "nextStopSeq": 8,                  // The number of next scheduled stop (8th stop of the day)
    "source": "vm",                    // How the data was recorded
    "type": "bus",                     // Vehicle type (bus, rail, etc.)
    "blockID": 248,                    // Groups multiple trips into blocks
    "signMessageLong": "FX2 Division To Gresham", // More description for message
    "lastLocID": 3397,                 // Stop ID of the last stop location
    "nextLocID": 13732,                
    "locationInScheduleDay": 49133,    // what time bus should be at a certain stop
    "routeSubType": "BRT",             // Subtype (e.g., Bus Rapid Transit)
    "newTrip": false,                  // true if new it's new trip, always false in data
    "longitude": -122.66883380836836,  
    "direction": 0,                    // Route direction (0, 1)
    "inCongestion": false,             // True if vehicle is stopped in traffic
    "routeNumber": 2,                  
    "bearing": 46,                     // Vehicle’s compass (0-360)
    "garage": "POWELL",                
    "tripID": "15644087",              
    "delay": -112,                     // measured in seconds, positive means early
    "extraBlockID": null,              
    "messageCode": 100,                // relates to message display
    "lastStopSeq": 7,                  // the last stop number in sequence of sstop
    "vehicleID": 4531,                 
    "time": 1760992845102,             // Timestamp when this was recorded by vehicle
    "offRoute": false                  // True if vehicle is off its scheduled route
  }

    

```
## Notable information from data documentation

* type - type of vehicle (bus or rail)
* longitude, latitude - location of vehicle
* inCongestion (Experimental) - *true* if vehicle not moving in traffic
* loadPercentage (Experimental) - estimate how many people riding
* vehicle_id - identifier
* time - time when position was recorded
* delay - positive means early, negative means late, measured in seconds

* bearing: which direction vehicle is going. (360 degrees)
* nextLocID - the previous stop
  
* routeNumber - Route number the vehicle is in (FX2, FX9)
* tripID - the planned trip from *trips.txt*
* garage -  Where bus originates from
* locationInScheduleDay - what time the bus should be at a certain point


## Creating DataFrame from csv

In [2]:
import pandas as pd
df = pd.read_csv('../data/vehicle_positions_full_raw.csv')

## Understanding the data

In [3]:
df.head(2)

,bearing,blockID,collection_timestamp,delay,direction,expires,extraBlockID,garage,inCongestion,lastLocID,...,routeNumber,routeSubType,serviceDate,signMessage,signMessageLong,source,time,tripID,type,vehicleID
0,200.0,243,1760817789,-1132,0,1760818026912,NaN,POWELL,False,7642.0,...,2,BRT,1760770800000,FX2 To Gresham,FX2 Division To Gresham,vm,1760817786912,15643905,bus,4530
1,269.0,248,1760817789,-499,1,1760818022926,NaN,POWELL,False,14241.0,...,2,BRT,1760770800000,FX2 To Portland,FX2 Division To Portland,vm,1760817782926,15644364,bus,4528


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 727199 entries, 0 to 727198
Data columns (total 31 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   bearing                727184 non-null  float64
 1   blockID                727199 non-null  int64  
 2   collection_timestamp   727199 non-null  int64  
 3   delay                  727199 non-null  int64  
 4   direction              727199 non-null  int64  
 5   expires                727199 non-null  int64  
 6   extraBlockID           1611 non-null    float64
 7   garage                 725588 non-null  object 
 8   inCongestion           562449 non-null  object 
 9   lastLocID              601971 non-null  float64
 10  lastStopSeq            601971 non-null  float64
 11  latitude               727199 non-null  float64
 12  loadPercentage         491051 non-null  float64
 13  locationInScheduleDay  727199 non-null  int64  
 14  longitude              727199 non-nu

### Setting up data

#### 1. Clean up with Date and Time

In [6]:
df_clean = df
df_clean['collection_timestamp'] = pd.to_datetime(df_clean['collection_timestamp'], unit='ms', errors='coerce')
df_clean['serviceDate'] = pd.to_datetime(df_clean['serviceDate'], unit='ms', errors='coerce')
df_clean['time'] = pd.to_datetime(df_clean['time'], unit='ms', errors='coerce')
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 727199 entries, 0 to 727198
Data columns (total 31 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   bearing                727184 non-null  float64       
 1   blockID                727199 non-null  int64         
 2   collection_timestamp   727199 non-null  datetime64[ns]
 3   delay                  727199 non-null  int64         
 4   direction              727199 non-null  int64         
 5   expires                727199 non-null  int64         
 6   extraBlockID           1611 non-null    float64       
 7   garage                 725588 non-null  object        
 8   inCongestion           562449 non-null  object        
 9   lastLocID              601971 non-null  float64       
 10  lastStopSeq            601971 non-null  float64       
 11  latitude               727199 non-null  float64       
 12  loadPercentage         491051 non-null  floa

Let's extract the day of the week and hour to make it easier to analyze

In [7]:
df['hour'] = df['collection_timestamp'].dt.hour
df['day'] = df['collection_timestamp'].dt.day_name()

### Non obvious columns

Some columns are not straightforward and will need more research to get better understanding.

#### routeSubType

In [8]:
# Check what RouteSubtype column is:
df['routeSubType'].unique()

array(['BRT', 'Bus', 'Light Rail', 'Shuttle'], dtype=object)

This shows that `routeSubType` gives more specification to vehicle info.  A BRT is "Bus Rapid Transit" and a "Bus" is the smaller old buses both are categorized as `bus` for the `type` column.

#### offRoute

Perhaps we can see if vehicles ever go off route, if not we can drop the entire column

In [9]:
df['offRoute'].value_counts()

offRoute
False    722640
True       4559
Name: count, dtype: int64

A small portion goes off route which will be interesting to anlyze later on.

#### Extra Block ID


In [ ]:
extra_block_rows = df[df['extraBlockID'].notna()]

# count where extraBlockID = blockID
same = (df['extraBlockID'] == df['blockID']).sum()
# count how many extrablocks there are
diff = df['extraBlockID'].notna().sum()

print(f"Extrablock id same as block id: {same}")
print(f"Total extrablock ids : {diff}")



Extrablocks id same: 1611
Total extrablockids : 1611


From the above code, we can see extrablockid and blockid is always the same.  So it might just be redundant information.

Let's look into how often extraBlockID appears

In [11]:
df['extraBlockID'].value_counts()

extraBlockID
767.0    413
734.0    259
702.0    225
701.0    213
768.0    200
736.0    147
769.0     76
737.0     50
770.0     28
Name: count, dtype: int64

The blockID starting with 700 seems to be the only one recording extrablock id which might be interesting case to study later.

#### New Trip is always false

In [12]:

print(df['newTrip'].value_counts())
print(df['offRoute'].value_counts())

newTrip
False    727199
Name: count, dtype: int64
offRoute
False    722640
True       4559
Name: count, dtype: int64


newTrip is always False, but sometimes buses go offRoute.

### Non important columns to drop

* vm - internal tool by trimet to label vehicle
* serviceDate - all the same, redundant since we know the date from our collection date
* newTrip - Was always false
* signMessage - redundant, signMessageLong gives more specific info
* expires - used by trimet to make sure data is up to date
* routeColor - redundant, route number should be sufficient to know color

After looking through more columns, 

### Checking for outliers

In [13]:
df.describe()

,bearing,blockID,collection_timestamp,delay,direction,expires,extraBlockID,lastLocID,lastStopSeq,latitude,...,longitude,messageCode,nextLocID,nextStopSeq,routeNumber,serviceDate,time,tripID,vehicleID,hour
count,727184.000000,727199.000000,727199,727199.000000,727199.000000,7.271990e+05,1611.000000,601971.000000,601971.000000,727199.000000,...,727199.000000,727199.000000,727199.000000,727199.000000,727199.000000,727199,727199,7.271990e+05,727199.000000,727199.0
mean,175.633082,5168.212412,1970-01-21 09:08:22.840238535,-128.287474,0.498400,1.760903e+12,740.400993,7563.460567,29.581302,45.507922,...,-122.664693,613.279281,7899.231099,24.523740,64.997089,2025-10-19 05:22:54.170210560,2025-10-19 19:40:25.021113344,1.562486e+07,2905.597924,9.0
min,0.000000,134.000000,1970-01-21 09:06:57.789000,-8595.000000,0.000000,1.760818e+12,701.000000,2.000000,1.000000,45.284999,...,-123.115567,13.000000,2.000000,1.000000,1.000000,2025-10-17 07:00:00,2025-10-18 19:57:06.241000,9.007170e+05,103.000000,9.0
25%,90.000000,1902.000000,1970-01-21 09:07:29.808000,-157.000000,0.000000,1.760850e+12,702.000000,4439.000000,11.000000,45.490587,...,-122.715628,316.000000,4651.000000,3.000000,19.000000,2025-10-18 07:00:00,2025-10-19 04:55:46.507000064,1.565053e+07,3024.000000,9.0
50%,180.000000,5473.000000,1970-01-21 09:08:26.001000,-30.000000,0.000000,1.760906e+12,736.000000,8045.000000,24.000000,45.517125,...,-122.662312,637.000000,8345.000000,17.000000,57.000000,2025-10-19 07:00:00,2025-10-19 20:33:20.299000064,1.565755e+07,3420.000000,9.0
75%,270.000000,8743.000000,1970-01-21 09:08:57.631000,0.000000,1.000000,1.760938e+12,767.000000,10028.000000,42.000000,45.530573,...,-122.578744,863.000000,10611.000000,38.000000,88.000000,2025-10-19 07:00:00,2025-10-20 05:19:29.446000128,1.566456e+07,3935.000000,9.0
max,359.000000,9767.000000,1970-01-21 09:09:46.457000,8893.000000,1.000000,1.760989e+12,770.000000,14661.000000,133.000000,45.639116,...,-122.330200,1094.000000,14661.000000,134.000000,293.000000,2025-10-20 07:00:00,2025-10-20 18:54:16.908000,1.567294e+07,4531.000000,9.0
std,102.343308,3249.810505,NaN,375.320404,0.499998,5.178153e+07,27.388769,4042.059391,24.164961,0.048849,...,0.127264,324.258516,4132.484296,24.947662,60.402103,NaN,NaN,6.911754e+05,1439.332417,0.0


In [14]:
print(f'max latitude position {df['latitude'].max()}')
print(f'min latitude position {df['latitude'].min()}')
print(f'max longitude position {df['longitude'].max()}')
print(f'min longitude position {df['longitude'].min()}')

max latitude position 45.63911567610055
min latitude position 45.284998982742216
max longitude position -122.3302
min longitude position -123.115567
